In [2]:
pip install -q gradio shap sentence-transformers faiss-cpu transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 54.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 91.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 4.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.

In [3]:
%%writefile app.py

# app.py
# Legal Judgment AI — Gradio (improved FLAN explanations with few-shot prompt)
# Requirements: gradio, transformers, sentence-transformers, faiss, shap, torch, matplotlib, pandas, numpy

import os, json, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import gradio as gr
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, T5Tokenizer, T5ForConditionalGeneration
import faiss, shap, warnings
from shap.maskers import Text

warnings.filterwarnings("ignore")

# -----------------------
# CONFIG / PATHS (update if different)
# -----------------------
MODEL_DIR = "/kaggle/input/nlp-stage-2/legalbert_classifier"         # fine-tuned Legal-BERT
FAIRNESS_DIR = "/kaggle/input/nlp-project-stage-4a-fairness"        # fairness csvs
FAISS_DIR = "/kaggle/input/nlp-project-stage-1"                     # faiss artifacts
CALIBRATED_DIR = "/kaggle/input/nlp-stage-2b"                       # optional pred_calibrated.csv
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------
# LOAD MODELS & ARTIFACTS
# -----------------------
print("Loading models and artifacts...")

# Legal-BERT
bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
bert_model.eval()
label_classes = np.load(os.path.join(MODEL_DIR, "label_classes.npy"), allow_pickle=True)
print("Loaded Legal-BERT, labels:", label_classes.tolist())

# FLAN-T5: try large then fallback to base
FLAN_NAMES = ["google/flan-t5-large", "google/flan-t5-base"]
t5_tokenizer = None
t5_model = None
for nm in FLAN_NAMES:
    try:
        print(f"Attempting to load {nm} ...")
        t5_tokenizer = T5Tokenizer.from_pretrained(nm)
        t5_model = T5ForConditionalGeneration.from_pretrained(nm).to(DEVICE)
        t5_model.eval()
        print("Loaded", nm)
        break
    except Exception as e:
        print(f"Could not load {nm}: {e}")
if t5_model is None:
    raise RuntimeError("Could not load any FLAN-T5 model. Install model files or change FLAN_NAMES.")

# FAISS + embeddings
faiss_index = None
id_to_meta = {}
embed_model = None
try:
    idx_path = os.path.join(FAISS_DIR, "faiss_index.bin")
    meta_path = os.path.join(FAISS_DIR, "id_to_meta.json")
    if os.path.exists(idx_path):
        faiss_index = faiss.read_index(idx_path)
    if os.path.exists(meta_path):
        with open(meta_path, "r") as fh:
            id_to_meta = json.load(fh)
    embed_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2").to(DEVICE)
    print("Loaded FAISS artifacts (if present)")
except Exception as e:
    print("FAISS load warning:", e)

# calibrated predictions (optional)
calibrated_df = None
calib_path = os.path.join(CALIBRATED_DIR, "pred_calibrated.csv")
if os.path.exists(calib_path):
    calibrated_df = pd.read_csv(calib_path)
    calibrated_df["text_clean"] = calibrated_df["text"].astype(str).str[:500].str.lower()
    print("Loaded calibrated predictions:", len(calibrated_df))

# -----------------------
# PREDICTION helper (calibration -> model -> temperature -> keyword override)
# -----------------------
def predict_case(text: str):
    text = str(text).strip()
    if not text:
        return "Unknown", 0.0, np.array([0.5, 0.5])

    # 1) calibrated lookup
    if calibrated_df is not None and len(calibrated_df):
        t_short = text[:300].lower().strip()
        exact = calibrated_df[calibrated_df["text_clean"] == t_short]
        if exact.shape[0] > 0:
            row = exact.iloc[0]
            p0 = float(row.get("Civil_prob", 0.5))
            p1 = float(row.get("Criminal_prob", 0.5))
            probs = np.array([p0, p1]); probs = probs / probs.sum()
            pred_idx = int(np.argmax(probs)); label = "Civil" if pred_idx == 0 else "Criminal"
            return label, float(probs[pred_idx]), probs
        contains = calibrated_df[calibrated_df["text_clean"].str.contains(t_short[:100], na=False)]
        if contains.shape[0] > 0:
            row = contains.iloc[0]
            p0 = float(row.get("Civil_prob", 0.5)); p1 = float(row.get("Criminal_prob", 0.5))
            probs = np.array([p0, p1]); probs = probs / probs.sum()
            pred_idx = int(np.argmax(probs)); label = "Civil" if pred_idx == 0 else "Criminal"
            return label, float(probs[pred_idx]), probs

    # 2) model logits (Legal-BERT)
    enc = bert_tokenizer(text, truncation=True, padding=True, return_tensors="pt", max_length=512)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = bert_model(**enc).logits.cpu().numpy()[0]
    probs = np.exp(logits) / np.sum(np.exp(logits))

    # 3) temperature scaling
    temperature = 1.2
    logits_safe = np.log(np.clip(probs, 1e-9, 1.0))
    scaled_logits = logits_safe / temperature
    rescaled = np.exp(scaled_logits) / np.sum(np.exp(scaled_logits))

    # 4) slight civil dampening
    civil_bias_factor = 0.95
    rescaled[0] *= civil_bias_factor
    rescaled = rescaled / rescaled.sum()

    # 5) criminal keyword override (pragmatic)
    criminal_keywords = [
        "section 302", "section 307", "section 376", "murder", "rape", "ipc", "convicted",
        "accused", "imprisonment", "sentence", "bail", "charge sheet", "criminal appeal", "session court"
    ]
    lower = text.lower()
    if any(k in lower for k in criminal_keywords):
        if rescaled[1] < 0.6:
            rescaled = np.array([0.3, 0.7]); rescaled = rescaled / rescaled.sum()

    pred_idx = int(np.argmax(rescaled))
    label = "Civil" if pred_idx == 0 else "Criminal"
    return label, float(rescaled[pred_idx]), rescaled

# -----------------------
# FLAN-T5 few-shot structured explanation
# -----------------------
# We provide 2 short few-shot examples in the prompt to guide output into: Reason / Law / Outcome
FEW_SHOT_EXAMPLES = [
    {
        "text": "The plaintiff sued for recovery of money under a sales contract. The seller failed to deliver goods and court ordered refund with interest.",
        "label": "Civil",
        "ex": "Reason: Seller failed to deliver agreed goods despite payment.\nLaw: Contract law — specific performance & restitution.\nOutcome: Court ordered refund with interest."
    },
    {
        "text": "The accused was convicted under Section 302 IPC for murder. The prosecution proved intentional killing and accused was sentenced to life imprisonment.",
        "label": "Criminal",
        "ex": "Reason: Intentional killing proved by prosecution evidence.\nLaw: Indian Penal Code, Section 302 (murder).\nOutcome: Conviction and sentence imposed by trial court."
    }
]

def make_flan_prompt(text: str, predicted_label: str):
    # build few-shot prompt
    prompt_lines = []
    prompt_lines.append("You are a concise legal summarizer. For the excerpt produce exactly three short labeled lines: 'Reason:', 'Law:', 'Outcome:'. Be factual; don't add opinions.")
    prompt_lines.append("")
    # add few-shot examples
    for ex in FEW_SHOT_EXAMPLES:
        prompt_lines.append("EXAMPLE CASE:")
        prompt_lines.append(ex["text"])
        prompt_lines.append("LABEL: " + ex["label"])
        prompt_lines.append("EXPLANATION:")
        prompt_lines.append(ex["ex"])
        prompt_lines.append("")
    prompt_lines.append("NOW EXPLAIN THE FOLLOWING CASE:")
    prompt_lines.append(text[:1200])
    prompt_lines.append("")
    prompt_lines.append(f"EXPECTED LABEL: {predicted_label}")
    prompt_lines.append("EXPLANATION:")
    return "\n".join(prompt_lines)

def flan_structured_explain(text: str, pred_label: str):
    prompt = make_flan_prompt(text, pred_label)
    # deterministic beam search (less copying) and low temperature
    enc = t5_tokenizer(prompt, return_tensors="pt", truncation=True, padding=True, max_length=1400).to(DEVICE)
    with torch.no_grad():
        out = t5_model.generate(**enc, num_beams=4, max_new_tokens=200, temperature=0.3, early_stopping=True)
    explanation = t5_tokenizer.decode(out[0], skip_special_tokens=True).strip()

    # Post-process: ensure three labelled lines Reason / Law / Outcome
    # Remove leading/trailing noise, split lines by newline or period heuristics
    lines = [ln.strip() for ln in explanation.splitlines() if ln.strip()]
    if len(lines) < 3:
        # try splitting by sentences
        sents = [s.strip() for s in explanation.replace("•", ".").split(".") if s.strip()]
        lines = sents[:3]
    # ensure prefixes
    targets = ["Reason:", "Law:", "Outcome:"]
    out_lines = []
    for i in range(3):
        raw = lines[i] if i < len(lines) else ""
        # if raw already contains one of the tags, keep it; else prefix with the expected tag
        if any(raw.lower().startswith(t.lower()) for t in targets):
            out_lines.append(raw)
        else:
            out_lines.append(f"{targets[i]} {raw}")
    # finally, trim each line to reasonable length (avoid echoing whole doc)
    out_lines = [ln if len(ln) <= 220 else ln[:217].rstrip() + "..." for ln in out_lines]
    return "\n".join(out_lines)

# -----------------------
# SHAP token heatmap (same as before)
# -----------------------
def shap_token_heatmap(text: str, target_class: int = 1):
    bert_model.eval()

    def predict_fn(batch_texts):
        if isinstance(batch_texts, np.ndarray):
            batch_texts = batch_texts.tolist()
        if isinstance(batch_texts, str):
            batch_texts = [batch_texts]
        batch_texts = [str(t) for t in batch_texts]
        enc = bert_tokenizer(batch_texts, truncation=True, padding=True, return_tensors="pt", max_length=256)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            logits = bert_model(**enc).logits
            probs = torch.softmax(logits, dim=1).cpu().numpy()
        return probs

    masker = Text(tokenizer=bert_tokenizer)
    explainer = shap.Explainer(predict_fn, masker)
    demo_text = text[:400]
    shap_vals = explainer([demo_text])

    vals = shap_vals[0].values
    if vals.ndim == 2:
        token_imp = vals[:, target_class]
    else:
        token_imp = vals
    tokens = shap_vals[0].data
    max_abs = np.max(np.abs(token_imp)) if len(token_imp) else 1.0
    if max_abs == 0:
        max_abs = 1.0
    norm = token_imp / max_abs

    fig, ax = plt.subplots(figsize=(10, 2.2))
    ax.axis("off")
    x, y = 0.01, 0.6
    for i, tok in enumerate(tokens):
        val = float(norm[i])
        cmap = plt.get_cmap("bwr")
        color = cmap((val + 1) / 2.0)
        txt = ax.text(x, y, tok + " ", fontsize=10,
                      bbox=dict(facecolor=color, edgecolor='none', pad=2.0, alpha=0.65),
                      verticalalignment='center')
        renderer = fig.canvas.get_renderer()
        bb = txt.get_window_extent(renderer=renderer)
        bb_data = bb.transformed(ax.transData.inverted())
        width = bb_data.x1 - bb_data.x0
        x += width + 0.007
        if x > 0.98:
            x = 0.01
            y -= 0.22
    import matplotlib as mpl
    cax = fig.add_axes([0.01, 0.02, 0.98, 0.08])
    cmap = mpl.cm.bwr
    normer = mpl.colors.Normalize(vmin=-max_abs, vmax=max_abs)
    cb = mpl.colorbar.ColorbarBase(cax, cmap=cmap, norm=normer, orientation='horizontal')
    cb.set_label('Token contribution (negative -> favors Civil, positive -> favors Criminal)')
    plt.tight_layout()
    return fig

# -----------------------
# Similar retrieval & fairness functions (unchanged)
# -----------------------
import re

def clean_case_text(raw_text: str):
    """
    Remove metadata, headers, and procedural noise from legal judgments.
    Keeps only the main content (facts, reasoning, decision).
    """
    text = str(raw_text)
    
    # Normalize spaces and line breaks
    text = re.sub(r"\s+", " ", text)

    # Remove typical headings / metadata phrases
    remove_patterns = [
        r"\bIN THE SUPREME COURT OF INDIA\b",
        r"\bREPORTABLE\b",
        r"\bJ U D G M E N T\b",
        r"\bJUDGMENT\b",
        r"\bO R D E R\b",
        r"\bORDER\b",
        r"\bCIVIL APPELLATE JURISDICTION\b",
        r"\bCRIMINAL APPELLATE JURISDICTION\b",
        r"\bCIVIL ORIGINAL JURISDICTION\b",
        r"\bCRIMINAL ORIGINAL JURISDICTION\b",
        r"\bSPECIAL LEAVE PETITION\b",
        r"\bWRIT PETITION\b",
        r"\bAPPEAL\b",
        r"\bAPPELLANT\b",
        r"\bRESPONDENT\b",
        r"\bPETITIONER\b",
        r"\bVERSUS\b",
        r"\bHON'?BLE\b",
        r"\bMR\.? JUSTICE\b",
        r"\bMS\.? JUSTICE\b",
        r"[\(\)\[\]\{\}•§©¶]+",  # punctuation junk
        r"[0-9]{1,3}\s*\."       # numbering like "1.", "2."
    ]
    for pat in remove_patterns:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)

    # Remove any long uppercase title chunks (metadata noise)
    text = re.sub(r"\b[A-Z\s]{8,}\b", "", text)

    # Trim whitespace and redundant dots
    text = re.sub(r"\.{3,}", ".", text)
    text = re.sub(r"\s{2,}", " ", text).strip()

    return text


def retrieve_similar(query: str, top_k: int = 3):
    """
    Retrieve top-k similar cases (cleaned content only).
    """
    if faiss_index is None or embed_model is None:
        return "⚠️ FAISS index or embedding model not available."

    q_emb = embed_model.encode([query], convert_to_numpy=True, device=DEVICE).astype("float32")
    faiss.normalize_L2(q_emb)
    D, I = faiss_index.search(q_emb, top_k)

    results = []
    for score, idx in zip(D[0], I[0]):
        key = str(idx)
        meta = id_to_meta.get(key, id_to_meta.get(idx, {}))

        # Retrieve main text content
        full_text = meta.get("text", "") or meta.get("case_text", "") or meta.get("snippet", "")
        full_text = clean_case_text(full_text)

        # Take a concise excerpt (first few sentences)
        sentences = re.split(r"(?<=[.!?]) +", full_text)
        preview = " ".join(sentences[:5]).strip()

        case_no = meta.get("case_no", "Unknown Case No.")
        results.append(f"**Case:** {case_no}\n**Similarity Score:** {score:.4f}\n\n{preview}...\n")

    return "\n---\n".join(results) if results else "No similar cases found."



def load_fairness():
    fair_path = os.path.join(FAIRNESS_DIR, "fairness_audit_results.csv")
    causal_path = os.path.join(FAIRNESS_DIR, "causal_fairness_report.csv")
    if not os.path.exists(fair_path):
        return None, "No fairness CSV found", pd.DataFrame()
    df = pd.read_csv(fair_path)
    plot_df = df[~df["Group"].str.contains("Overall", na=False)].copy()
    fig, ax = plt.subplots(figsize=(9, 5))
    colors = ["#4caf50" if abs(v) <= 10 else "#ff9800" if abs(v) <= 50 else "#f44336" for v in plot_df["Bias(%)"]]
    ax.barh(plot_df["Group"], plot_df["Bias(%)"], color=colors, edgecolor="black")
    ax.axvline(0, color='k')
    ax.set_xlabel("Bias (Civil % − Criminal %)")
    ax.set_title("Fairness Bias by Group")
    plt.tight_layout()
    summary = f"Unweighted Jain's Index = {((np.sum(plot_df['Civil_Rate'])**2)/(len(plot_df['Civil_Rate'])*np.sum(plot_df['Civil_Rate']**2))):.3f}"
    causal_df = pd.read_csv(causal_path) if os.path.exists(causal_path) else pd.DataFrame()
    return fig, summary, causal_df.head(12)

# -----------------------
# Gradio callbacks
# -----------------------
def classify_and_explain(text):
    if not text or not str(text).strip():
        return "Please paste judgment text.", "", None
    label, conf, probs = predict_case(text)
    # better structured explanation using few-shot prompt
    try:
        explanation = flan_structured_explain(text, label)
    except Exception as e:
        explanation = f"FLAN explanation failed: {e}"

    target = int(np.argmax(probs))
    try:
        fig = shap_token_heatmap(text, target_class=target)
    except Exception as e:
        fig, ax = plt.subplots(figsize=(8,2))
        ax.axis("off")
        ax.text(0.5, 0.5, f"SHAP failed: {str(e)}", ha="center", va="center", color="red")
        plt.tight_layout()

    prob_str = f"Civil={probs[0]:.3f}, Criminal={probs[1]:.3f}"
    header = f"Prediction: **{label}** (conf={conf:.3f}) — {prob_str}"
    return header, explanation, fig

# -----------------------
# Gradio UI (unchanged structure)
# -----------------------
with gr.Blocks(title="Legal Judgment AI — Classification & Explainability") as demo:
    gr.Markdown("## ⚖️ Legal Judgment AI — Classification, Explainability & Fairness")
    with gr.Tab("Classify & Explain"):
        txt = gr.Textbox(label="Paste judgment text", lines=8, placeholder="Paste judgment excerpt here...")
        go = gr.Button("Classify + Explain")
        out_header = gr.Markdown()
        out_explain = gr.Textbox(label="FLAN-T5 structured explanation (Reason / Law / Outcome)", lines=4)
        out_shap = gr.Plot()
        go.click(classify_and_explain, inputs=txt, outputs=[out_header, out_explain, out_shap])

    with gr.Tab("Similar Case Retrieval"):
        q = gr.Textbox(label="Enter query", lines=2, placeholder="e.g., bail in criminal case")
        k = gr.Slider(minimum=1, maximum=10, value=3, step=1, label="Top K Results")
        btn_search = gr.Button("Retrieve Similar Cases")
        result_md = gr.Markdown(label="Top Results (click + expand for full text)")
        btn_search.click(retrieve_similar, inputs=[q, k], outputs=result_md)


    with gr.Tab("Fairness Dashboard"):
        fbtn = gr.Button("Load fairness CSVs")
        fplot = gr.Plot()
        fsummary = gr.Textbox()
        ftable = gr.Dataframe()
        fbtn.click(load_fairness, inputs=None, outputs=[fplot, fsummary, ftable])

   
demo.launch(debug=False, share=True)


Writing app.py


In [ ]:
!python app.py


2025-11-11 12:41:48.168152: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762864908.586315      92 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762864908.716603      92 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
Loading models and artifacts...
Loaded Legal-BERT, labels: ['Civil', 'Crimina